# US bysykkel data

- data is from 2024
- Contact person is Christoffer Bakken Åkre (US)

## Imports

In [906]:
import pandas as pd
from pathlib import Path
import os
import numpy as np
from geopy.distance import geodesic
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

## Load cleaned data

In [907]:
# Directories setup
notebook_dir = Path(os.getcwd())
cleaned_trip_files_dir = notebook_dir / 'trip_data' / 'extracted_data'
cleaned_damage_files_dir = notebook_dir / 'maintenance_data' / 'processed_maintenance'

_trip_files = sorted(cleaned_trip_files_dir.glob('*.csv'))
_maintenance_files = sorted(cleaned_damage_files_dir.glob('*.csv'))

print(f"Found {len(_trip_files)} CSV files in {cleaned_trip_files_dir}")
print(f"Found {len(_maintenance_files)} CSV files in {cleaned_damage_files_dir}")

# --- Load Trip Data (Keeping the logs) ---
trip_dfs = []
for file in _trip_files:
    print(f"Loading {file.name}...")
    df = pd.read_csv(file)
    trip_dfs.append(df)

# Combine into one master trip dataframe
trip_data = pd.concat(trip_dfs, ignore_index=True) if trip_dfs else pd.DataFrame()

# --- Load Maintenance Data (Separating into Damage, Maint, Repair) ---
damage_list = []
maint_list = []
repair_list = []

for file in _maintenance_files:
    print(f"Loading {file.name}...")
    df = pd.read_csv(file)
    
    fname = file.name.lower()
    if 'damage' in fname:
        damage_list.append(df)
    elif 'repair' in fname:
        repair_list.append(df)
    else:
        # Assuming files not marked damage/repair are general maintenance
        maint_list.append(df)

# Create the separate DataFrames
damage_data = pd.concat(damage_list, ignore_index=True) if damage_list else pd.DataFrame()
maintenance_data = pd.concat(maint_list, ignore_index=True) if maint_list else pd.DataFrame()
repair_data = pd.concat(repair_list, ignore_index=True) if repair_list else pd.DataFrame()

print("-" * 30)
print(f"Trips loaded: {len(trip_data)} rows")
print(f"Damage logs:  {len(damage_data)} rows")
print(f"Maint logs:   {len(maintenance_data)} rows")
print(f"Repair logs:  {len(repair_data)} rows")



Found 1 CSV files in c:\Users\Minamsj\FOMOsim\policies\sjovik_sund\US_data\trip_data\extracted_data
Found 3 CSV files in c:\Users\Minamsj\FOMOsim\policies\sjovik_sund\US_data\maintenance_data\processed_maintenance
Loading cleaned_trip_data.csv...
Loading cleaned_damage_data.csv...
Loading cleaned_maintenance_data.csv...
Loading cleaned_repair_data.csv...
------------------------------
Trips loaded: 151278 rows
Damage logs:  12845 rows
Maint logs:   22786 rows
Repair logs:  9839 rows


## Split damages into mechanical and system errors

In [908]:
mechanical_damages = [
    'Bremse(r)',                   # Brakes (Physical wear)
    'Brake(s) need adjustment',    # Brakes (Cable stretch/wear)
    'Hjul',                        # Wheels (Impact/wear)
    'Lite luft',                   # Low Air (Punctures/Valve leaks)
    'Gir',                         # Gears (Mechanical wear)
    'Belte / Belt',                # Drive Belt (Friction wear)
    'Pedaler',                     # Pedals (Bearings/Impact)
    'Cranck bearing/bottom bracket ', # Cranks (Bearing fatigue)
    'Styre',                       # Handlebar (Impact/Fatigue)
    'Styrelager',                  # Headset Bearings (Vibration)
    'Sete',                        # Seat (Physical damage)
    'Setepinneklemme',             # Seat Clamp (Mechanical stress)
    'Støtte',                      # Kickstand (Spring/Hinge wear)
    'Skjerm(er)',                  # Fenders (Vibration/Impact)
    'Ringeklokke',                 # Bell (Mechanical spring)
    'Basket',                      # Basket (Load/Impact)
    'Lys',                         # Lights (Vibration/Wiring fatigue)
    'Frame'                        # Frame (Structural stress)
]

system_damages = [
    'Unresponsive Controller',     # Connectivity/Firmware (The big one)
    'Too many quick returns',      # User Behavior/System Flag
    'Unauthorized Trip',           # System/Theft Flag
    'Lock & unlock',               # Smart Lock Actuator/Comms
    'Lås',                         # Smart Lock Hardware
    'Console',                     # Dashboard Electronics
    'GPS',                         # Connectivity/Module
    'Battery',                     # Charging/BMS (Time + Cycles)
    'Vandalism'                    # External Factor (Scales with Time on Street)
]

In [909]:
damage_mechanical = damage_data[damage_data['damage_type_name'].isin(mechanical_damages)].copy()
damage_system = damage_data[damage_data['damage_type_name'].isin(system_damages)].copy()

print("--- SPLIT RESULTS ---")
print(f"Total Original Rows:   {len(damage_data):,}")
print(f"Mechanical Rows (Usage): {len(damage_mechanical):,}  (Use for Spare Parts Model)")
print(f"System Rows (Time):    {len(damage_system):,}  (Use for Reliability Model)")

lost_rows = len(damage_data) - (len(damage_mechanical) + len(damage_system))
if lost_rows > 0:
    print(f"\n[WARNING] {lost_rows} rows were not categorized!")
    uncategorized = damage_data[~damage_data['damage_type_name'].isin(mechanical_damages + system_damages)]
    print("Uncategorized types found:", uncategorized['damage_type_name'].unique())
else:
    print("\n[SUCCESS] All rows successfully categorized.")

--- SPLIT RESULTS ---
Total Original Rows:   12,845
Mechanical Rows (Usage): 6,126  (Use for Spare Parts Model)
System Rows (Time):    6,719  (Use for Reliability Model)

[SUCCESS] All rows successfully categorized.


# Poisson regression model

## Aggregate the data

In [910]:
print(tabulate(trip_data.head(200), 
               headers='keys',    
               tablefmt='psql', 
               showindex=False,
               floatfmt=".2f"))

+-----------+--------------+---------------------+-------------------------+-------------------------+--------------+--------------------------+----------------------------+-------------------------------+------------------------+--------------------------+-----------------------------+----------------------+------------------------------+
|   trip_id |   vehicle_id |   position_accuracy | trip_started_at         | trip_ended_at           | trip_state   |   trip_start_dock_number |   trip_start_dock_group_id | trip_start_dock_group_title   |   trip_end_dock_number |   trip_end_dock_group_id | trip_end_dock_group_title   |   segment_distance_m |   total_trip_distance_meters |
|-----------+--------------+---------------------+-------------------------+-------------------------+--------------+--------------------------+----------------------------+-------------------------------+------------------------+--------------------------+-----------------------------+----------------------+------

In [911]:
# --- STEP 1: Aggregate Trips (The 'Exposure' or Denominator) ---
# We sum up the distance traveled by each bike in each month
trip_exposure = (
    trip_data[trip_data['is_legitimate_trip']]
    .set_index('trip_started_at')
    .groupby([pd.Grouper(freq='MS'), 'vehicle_id'])
    .agg({'total_trip_distance_meters': 'sum'})
    .reset_index()
    .rename(columns={'trip_started_at': 'month_start', 'total_trip_distance_meters': 'total_distance_m'})
)

# --- STEP 2: Aggregate Damage (The 'Target' or Numerator) ---
# We count how many mechanical damages occurred for each bike in each month
mech_counts = (
    damage_mechanical  # Using your previously filtered mechanical dataframe
    .set_index('created_at')
    .groupby([pd.Grouper(freq='MS'), 'vehicle_id'])
    .size()
    .reset_index(name='damage_count')
    .rename(columns={'created_at': 'month_start'})
)

# --- STEP 3: The Final Merge (Creating df_model_mech) ---
# We join the usage and damages together. If a bike drove but didn't break, damage is 0.
df_model_mech = pd.merge(trip_exposure, mech_counts, on=['month_start', 'vehicle_id'], how='left')

# Fill missing values and convert to KM
df_model_mech['damage_count'] = df_model_mech['damage_count'].fillna(0).astype(int)
df_model_mech['distance_km'] = df_model_mech['total_distance_m'] / 1000.0

# Filter out rows with 0 distance to avoid math errors (NaNs) in the model
df_model_mech = df_model_mech[df_model_mech['distance_km'] > 0].copy()

print(f"[SUCCESS] df_model_mech created with {len(df_model_mech)} rows.")

# --- 4. PREPARE VEHICLE METADATA ---
print("\n--- STEP 4: PREPARING METADATA ---")
v_dmg = damage_data[['vehicle_id', 'vehicle_category', 'asset_model_id']]
v_mnt = maintenance_data[['vehicle_id', 'vehicle_category', 'asset_model_id']]
v_rpr = repair_data[['vehicle_id', 'vehicle_category', 'asset_model_id']]
vehicle_features = pd.concat([v_dmg, v_mnt, v_rpr]).drop_duplicates('vehicle_id')

# --- 5. BUILD MODEL A: MECHANICAL (DISTANCE BASED) ---
print("\n" + "="*50)
print("--- STEP 5: BUILDING MECHANICAL DATASET ---")

df_model_mech = pd.merge(trip_exposure, mech_counts, on=['month_start', 'vehicle_id'], how='left')
df_model_mech = df_model_mech.merge(vehicle_features, on='vehicle_id', how='left')

df_model_mech['damage_count'] = df_model_mech['damage_count'].fillna(0).astype(int)
df_model_mech['distance_km'] = df_model_mech['total_distance_m'] / 1000.0

# Filter out months with zero valid distance
df_model_mech = df_model_mech[df_model_mech['distance_km'] > 0].copy()

print(f"Final Mechanical Rows for Model: {len(df_model_mech):,}")

# DEBUG: Preview short but valid months
short_valid = df_model_mech[df_model_mech['distance_km'] < 1.0].head(10)
print(tabulate(short_valid[['month_start', 'vehicle_id', 'distance_km', 'trip_count', 'damage_count']], 
                headers='keys', tablefmt='psql', showindex=False))

# --- 6. BUILD MODEL B: SYSTEM (TIME BASED) ---
print("\n" + "="*50)
print("--- STEP 6: BUILDING SYSTEM DATASET ---")

# For the system model, we use the full trip_exposure (including all active months)
df_model_sys = pd.merge(trip_exposure, sys_counts, on=['month_start', 'vehicle_id'], how='left')
df_model_sys = df_model_sys.merge(vehicle_features, on='vehicle_id', how='left')

df_model_sys['damage_count'] = df_model_sys['damage_count'].fillna(0).astype(int)
df_model_sys['exposure_months'] = 1.0 

print(f"Final System Rows for Model: {len(df_model_sys):,}")
print("="*50)

KeyError: 'is_legitimate_trip'

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import gammaln

# --- Helper: Negative Binomial Log-PMF for Mixture Fitting ---
def nb_logpmf(k, mu, alpha):
    """
    Log-Probability Mass Function for Negative Binomial.
    Used for numerical stability during optimization.
    """
    if alpha <= 1e-6: # Fallback to Poisson if dispersion is near zero
        return -mu + k * np.log(mu + 1e-9) - gammaln(k + 1)
    
    r = 1.0 / alpha
    p = r / (r + mu)
    return (gammaln(k + r) - gammaln(k + 1) - gammaln(r) + 
            r * np.log(p) + k * np.log(1 - p + 1e-9))

# --- STEP 7: FIT GENERALIZED NEGBIN MIXTURE PARAMETERS ---
print("\n" + "="*50)
print("--- STEP 7: EXTRACTING NEGBIN FLEET DNA ---")

def fit_mixture_logic_nb(df):
    """
    Fits a Two-Component Negative Binomial Mixture.
    Separates baseline wear (Healthy) from catastrophic failure loops (Lemons).
    """
    counts = df['damage_count'].values
    dist = df['distance_km'].values

    def nll(params):
        # params: [pi, rate_h, alpha_h, rate_l, alpha_l]
        pi, lh, ah, ll, al = params
        
        # Physical Constraints
        if not (0.01 < pi < 0.99 and lh > 0 and ah > 0 and ll > 0 and al > 0):
            return 1e15
        
        # Expected values (mu) based on trip distance
        mu_h = (lh / 100.0) * dist
        mu_l = (ll / 100.0) * dist
        
        # Probability of the data point belonging to either group
        prob_h = np.exp(nb_logpmf(counts, mu_h, ah))
        prob_l = np.exp(nb_logpmf(counts, mu_l, al))
        
        # Total Log-Likelihood
        return -np.sum(np.log(pi * prob_h + (1 - pi) * prob_l + 1e-12))

    # Initial guess: 80% Healthy (0.02 rate), 20% Lemon (5.0 rate)
    # We use a 0.5 alpha for healthy and 2.0 for lemons as a starting point
    initial_guess = [0.80, 0.02, 0.5, 5.0, 2.0]
    res = minimize(nll, initial_guess, method='Nelder-Mead')
    return res.x

# Extract the 5 parameters of your fleet's DNA
pi_f, lh_f, ah_f, ll_f, al_f = fit_mixture_logic_nb(df_model_mech)

print(f"Generalized NegBin DNA:")
print(f" -> Healthy Group: {pi_f*100:.1f}% (Rate: {lh_f:.3f}/100km, Alpha: {ah_f:.2f})")
print(f" -> Lemon Group:   {(1-pi_f)*100:.1f}% (Rate: {ll_f:.3f}/100km, Alpha: {al_f:.2f})")

# --- STEP 8: THE SIMULATOR AGENT (ADVANCED) ---
class SimulatedBike:
    """
    A 'Digital Twin' using Negative Binomial logic.
    Captures both systemic degradation and random 'clumpy' failures.
    """
    def __init__(self, pi, lh, ah, ll, al):
        # 1. Stochastic Quality Assignment
        if np.random.random() < pi:
            self.state = "Healthy"
            self.base_rate = lh / 100.0
            self.alpha = ah
        else:
            self.state = "Lemon"
            self.base_rate = ll / 100.0
            self.alpha = al
            
        self.odometer_km = 0.0
        self.wear_coefficient = 0.00005 # Rate increases with mileage

    def calculate_current_mu(self, trip_km):
        # 2. Degradation Logic (Linear wear-and-tear)
        current_rate = self.base_rate * (1 + (self.wear_coefficient * self.odometer_km))
        return current_rate * trip_km

    def run_trip(self, trip_km):
        mu = self.calculate_current_mu(trip_km)
        
        # 3. Negative Binomial Failure Generator
        # Convert mu and alpha to n (successes) and p (probability) for numpy
        n = 1.0 / self.alpha
        p = n / (n + mu)
        
        # Returns number of damages sustained during this trip
        damage_count = np.random.negative_binomial(n, p)
        
        self.odometer_km += trip_km
        return damage_count

print("\n--- STEP 8: ADVANCED SIMULATOR READY ---")


--- STEP 7: EXTRACTING FLEET DNA FOR SIMULATOR ---
Generalized Fleet DNA:
 -> Healthy Pop: 71.2% (Rate: 0.016/100km)
 -> Lemon Pop:   28.8% (Rate: 5.309/100km)

--- STEP 8: SIMULATOR READY ---
You can now instantiate bikes based on the extracted DNA parameters.


## Identifying the worst bikes  

In [ ]:
from scipy.optimize import minimize
from scipy.stats import poisson
import numpy as np

# --- 1. GENERALIZED PARAMETER EXTRACTION (THE DNA) ---
def fit_fleet_mixture(df):
    """Fits a Two-Component Poisson Mixture to find generalized rates."""
    counts = df['damage_count'].values
    dist = df['distance_km'].values
    
    def nll(params):
        pi, lam_h, lam_l = params
        if not (0.001 < pi < 0.999 and lam_h > 0 and lam_l > 0): return 1e10
        # P(x) = pi * Poisson(Healthy) + (1-pi) * Poisson(Lemon)
        prob_h = poisson.pmf(counts, (lam_h/100) * dist)
        prob_l = poisson.pmf(counts, (lam_l/100) * dist)
        return -np.sum(np.log(pi * prob_h + (1 - pi) * prob_l + 1e-9))

    res = minimize(nll, [0.98, 0.04, 15.0], method='Nelder-Mead')
    return res.x

# Fit the generalized parameters
pi_h, lam_h, lam_l = fit_fleet_mixture(df_model_mech)

# --- 2. SPECIFIC VEHICLE AGGREGATION ---
fleet_avg_rate = (df_model_mech['damage_count'].sum() / df_model_mech['distance_km'].sum()) * 100

bike_summary = df_model_mech.groupby('vehicle_id').agg({
    'damage_count': 'sum',
    'distance_km': 'sum',
    'month_start': 'count'
}).reset_index()

bike_summary['failure_rate_100km'] = (bike_summary['damage_count'] / bike_summary['distance_km']) * 100
bike_summary['vs_avg'] = bike_summary['failure_rate_100km'] / fleet_avg_rate

# Filter and Sort Lemons
lemons = bike_summary[bike_summary['distance_km'] > 10].sort_values('failure_rate_100km', ascending=False).head(20)

# --- 3. PRINT RESULTS ---
print("\n" + "="*50)
print("PART 1: GENERALIZED FLEET DNA (For Simulator)")
print("="*50)
print(f"Healthy Population Probability: {pi_h*100:.1f}%")
print(f"Healthy Failure Rate:           {lam_h:.4f} per 100km")
print(f"Lemon Population Probability:   {(1-pi_h)*100:.1f}%")
print(f"Lemon Failure Rate:             {lam_l:.2f} per 100km")

print("\n" + "="*85)
print("PART 2: TOP 20 'LEMONS' IN CURRENT FLEET")
print("="*85)
print(f"{'Vehicle ID':<15} | {'Damages':<8} | {'Tot KM':<10} | {'Rate/100km':<12} | {'X Worse than Avg'}")
print("-"*85)
for _, row in lemons.iterrows():
    print(f"{row['vehicle_id']:<15} | {row['damage_count']:<8.0f} | {row['distance_km']:<10.1f} | {row['failure_rate_100km']:<12.2f} | {row['vs_avg']:.1f}x")


PART 1: GENERALIZED FLEET DNA (For Simulator)
Healthy Population Probability: 71.2%
Healthy Failure Rate:           0.0158 per 100km
Lemon Population Probability:   28.8%
Lemon Failure Rate:             5.31 per 100km

PART 2: TOP 20 'LEMONS' IN CURRENT FLEET
Vehicle ID      | Damages  | Tot KM     | Rate/100km   | X Worse than Avg
-------------------------------------------------------------------------------------
62804.0         | 11       | 28.4       | 38.67        | 910.7x
261.0           | 5        | 13.1       | 38.17        | 899.0x
4259.0          | 5        | 16.6       | 30.17        | 710.5x
60.0            | 12       | 39.9       | 30.08        | 708.6x
127.0           | 8        | 27.8       | 28.73        | 676.6x
62956.0         | 43       | 282.7      | 15.21        | 358.2x
29613.0         | 5        | 36.4       | 13.72        | 323.2x
290.0           | 17       | 125.8      | 13.51        | 318.3x
62800.0         | 13       | 97.3       | 13.36        | 314.6x
133

In [ ]:
# 1. Calculate the fleet-wide baseline (failures per 100km)
fleet_avg_rate = (df_model_mech['damage_count'].sum() / df_model_mech['distance_km'].sum()) * 100

# 2. Group by Vehicle ID to see cumulative performance
bike_summary = df_model_mech.groupby('vehicle_id').agg({
    'damage_count': 'sum',
    'distance_km': 'sum',
    'month_start': 'count' # How many months they were active
}).reset_index()

# 3. Calculate individual rates and "Pain Contribution"
bike_summary['failure_rate_100km'] = (bike_summary['damage_count'] / bike_summary['distance_km']) * 100
bike_summary['vs_avg'] = bike_summary['failure_rate_100km'] / fleet_avg_rate

# 4. Sort by the most "Extreme" bikes
# We filter for bikes with at least some distance to avoid division by zero
lemons = bike_summary[bike_summary['distance_km'] > 10].sort_values('failure_rate_100km', ascending=False).head(20)

print(f"\nFLEET BASELINE: {fleet_avg_rate:.2f} failures per 100km")
print("="*85)
print(f"{'Vehicle ID':<15} | {'Damages':<8} | {'Tot KM':<10} | {'Rate/100km':<12} | {'X Worse than Avg'}")
print("-"*85)
for _, row in lemons.iterrows():
    print(f"{row['vehicle_id']:<15} | {row['damage_count']:<8.0f} | {row['distance_km']:<10.1f} | {row['failure_rate_100km']:<12.2f} | {row['vs_avg']:.1f}x")


FLEET BASELINE: 0.04 failures per 100km
Vehicle ID      | Damages  | Tot KM     | Rate/100km   | X Worse than Avg
-------------------------------------------------------------------------------------
62804.0         | 11       | 28.4       | 38.67        | 910.7x
261.0           | 5        | 13.1       | 38.17        | 899.0x
4259.0          | 5        | 16.6       | 30.17        | 710.5x
60.0            | 12       | 39.9       | 30.08        | 708.6x
127.0           | 8        | 27.8       | 28.73        | 676.6x
62956.0         | 43       | 282.7      | 15.21        | 358.2x
29613.0         | 5        | 36.4       | 13.72        | 323.2x
290.0           | 17       | 125.8      | 13.51        | 318.3x
62800.0         | 13       | 97.3       | 13.36        | 314.6x
133.0           | 19       | 160.9      | 11.81        | 278.1x
4051.0          | 2        | 19.4       | 10.30        | 242.6x
224.0           | 6        | 62.1       | 9.66         | 227.5x
121.0           | 6        | 70

## Plotting distibutions

## Statistical modelling Poisson vs NegBinomial distribution

In [ ]:
import statsmodels.api as sm
from tabulate import tabulate
import numpy as np
from scipy.optimize import minimize
from scipy.special import gammaln

def nb_logpmf(k, mu, alpha):
    """
    Manual Log-PMF for Negative Binomial (to use in Mixture models).
    Uses the parameterization: Var = mu + alpha * mu^2
    """
    if alpha == 0: # Becomes Poisson
        return -mu + k * np.log(mu) - gammaln(k + 1)
    
    # Standard NB conversion to r, p
    r = 1.0 / alpha
    p = r / (r + mu)
    
    res = (gammaln(k + r) - gammaln(k + 1) - gammaln(r) + 
           r * np.log(p) + k * np.log(1 - p))
    return res

def robust_compare(df, category_name, offset_col, alpha_val=1.0, include_mixtures=False):
    y = df['damage_count'].values
    dist = df[offset_col].values
    X = sm.add_constant(np.ones(len(df)))
    offset = np.log(dist)

    # 1. Negative Binomial (GLM)
    model_nb = sm.GLM(y, X, offset=offset, 
                      family=sm.families.NegativeBinomial(alpha=alpha_val)).fit()
    
    # 2. Poisson (GLM)
    model_po = sm.GLM(y, X, offset=offset, 
                      family=sm.families.Poisson()).fit()

    rows = [
        [category_name, f"NegBin (α={alpha_val})", model_nb.llf, model_nb.aic, model_nb.pearson_chi2, model_nb.df_resid, model_nb.pearson_chi2 / model_nb.df_resid],
        ["", "Poisson", model_po.llf, model_po.aic, model_po.pearson_chi2, model_po.df_resid, model_po.pearson_chi2 / model_po.df_resid]
    ]

    if include_mixtures:
        # 3. NegBin Mixture (The "Lemon" Model)
        # Params: [pi, rate_h, alpha_h, rate_l, alpha_l]
        def mixture_nll(params):
            pi, lh, ah, ll, al = params
            if not (0.01 < pi < 0.99 and lh > 0 and ah > 0 and ll > 0 and al > 0): return 1e15
            
            # Means for this specific trip distance
            mu_h = (lh / 100.0) * dist
            mu_l = (ll / 100.0) * dist
            
            log_prob_h = nb_logpmf(y, mu_h, ah)
            log_prob_l = nb_logpmf(y, mu_l, al)
            
            # Log-Sum-Exp trick for numerical stability
            ll_total = np.log(pi * np.exp(log_prob_h) + (1 - pi) * np.exp(log_prob_l) + 1e-12)
            return -np.sum(ll_total)

        # Initial guess based on your data findings
        res = minimize(mixture_nll, [0.98, 0.04, 0.5, 15.0, 2.0], method='Nelder-Mead')
        pi_f, lh_f, ah_f, ll_f, al_f = res.x
        
        mix_llf = -res.fun
        mix_aic = 2*5 - 2*mix_llf # 5 parameters
        
        rows.append(["", "NegBin-Mixture (Lemon)", mix_llf, mix_aic, "N/A", len(df)-5, "N/A"])
        
        # Store these globally or return them for the simulator
        global FLEET_DNA
        FLEET_DNA = {'pi': pi_f, 'lh': lh_f, 'ah': ah_f, 'll': ll_f, 'al': al_f}

    return rows

# Generate Results
results = robust_compare(df_model_mech, "Mechanical (KM)", "distance_km", alpha_val=2.0, include_mixtures=True)

print(tabulate(results, headers=["Category", "Model Type", "Log-Likelihood", "AIC", "Pearson Chi2", "df Resid", "Dispersion"], tablefmt="psql"))

+-----------------+------------------------+------------------+---------+--------------------+------------+--------------------+
| Category        | Model Type             |   Log-Likelihood |     AIC | Pearson Chi2       |   df Resid | Dispersion         |
|-----------------+------------------------+------------------+---------+--------------------+------------+--------------------|
| Mechanical (KM) | NegBin (α=2.0)         |         -8472.94 | 16947.9 | 201365.06740277453 |       4604 | 43.73698249408656  |
|                 | Poisson                |        -19573.6  | 39149.2 | 8263466.73982      |       4604 | 1794.8450781537792 |
|                 | NegBin-Mixture (Lemon) |         -7154.33 | 14318.7 | N/A                |       4600 | N/A                |
+-----------------+------------------------+------------------+---------+--------------------+------------+--------------------+


In [ ]:
import statsmodels.api as sm
from tabulate import tabulate
import numpy as np

def robust_compare(df, category_name, offset_col, alpha_val=1.0):
    """
    Runs Poisson and NegBinomial with extra statistical fields for diagnostic.
    """
    y = df['damage_count']
    X = sm.add_constant(np.ones(len(df)))
    offset = np.log(df[offset_col])

    # 1. Negative Binomial (GLM)
    model_nb = sm.GLM(y, X, offset=offset, 
                      family=sm.families.NegativeBinomial(alpha=alpha_val)).fit()
    
    # 2. Poisson
    model_po = sm.GLM(y, X, offset=offset, 
                      family=sm.families.Poisson()).fit()

    return [
        [
            category_name, 
            f"NegBin (α={alpha_val})", 
            model_nb.llf, 
            model_nb.aic, 
            model_nb.pearson_chi2, 
            model_nb.df_resid,
            model_nb.pearson_chi2 / model_nb.df_resid
        ],
        [
            "", 
            "Poisson", 
            model_po.llf, 
            model_po.aic, 
            model_po.pearson_chi2, 
            model_po.df_resid,
            model_po.pearson_chi2 / model_po.df_resid
        ]
    ]

# Generate Results
results = []
results.extend(robust_compare(df_model_mech, "Mechanical (KM)", "distance_km", alpha_val=2.0))
results.extend(robust_compare(df_model_sys, "System (Months)", "exposure_months", alpha_val=1.5))

headers = ["Category", "Model Type", "Log-Likelihood", "AIC", "Pearson Chi2", "df Resid", "Dispersion"]
print("\n" + "="*110)
print("MODEL COMPARISON")
print("="*110)
print(tabulate(results, headers=headers, tablefmt="psql", floatfmt=".3f"))


MODEL COMPARISON
+-----------------+----------------+------------------+-----------+----------------+------------+--------------+
| Category        | Model Type     |   Log-Likelihood |       AIC |   Pearson Chi2 |   df Resid |   Dispersion |
|-----------------+----------------+------------------+-----------+----------------+------------+--------------|
| Mechanical (KM) | NegBin (α=2.0) |        -8472.942 | 16947.885 |     201365.067 |   4604.000 |       43.737 |
|                 | Poisson        |       -19573.613 | 39149.226 |    8263466.740 |   4604.000 |     1794.845 |
| System (Months) | NegBin (α=1.5) |        -6700.239 | 13402.478 |       5554.818 |   4630.000 |        1.200 |
|                 | Poisson        |        -8040.447 | 16082.894 |      14763.268 |   4630.000 |        3.189 |
+-----------------+----------------+------------------+-----------+----------------+------------+--------------+
